In [0]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("Time Series").getOrCreate()

df = spark.table("workspace.default.gold_anime")
display(df)

In [0]:
import pandas as pd
import matplotlib.pyplot as plt

ts_df = spark.table("workspace.default.gold_anime").toPandas()
print(ts_df.head())

In [0]:
ts_df = ts_df[ts_df["year"] < 2026]
scored = ts_df[ts_df["score"] > 0].copy()

**T1 — Production Growth**

In [0]:
yearly = ts_df.groupby("year")["title"].count().reset_index()
yearly.columns = ["year", "count"]

yearly["yoy_growth"] = yearly["count"].pct_change() * 100
print("Yearly Production Growth\n")
print(yearly)

**Insights**<br>
The industry shows cyclical growth with sharp expansions followed by corrections.

**T2 — Rolling Production Trend**

In [0]:
yearly["rolling"] = yearly["count"].rolling(3).mean()

plt.figure(figsize=(10,5))

plt.plot(yearly["year"], yearly["count"], label="Annual")
plt.plot(yearly["year"], yearly["rolling"], linestyle="--", label="3Y Avg")

plt.title("Anime Production Over Time")
plt.xlabel("Year")
plt.ylabel("Count")
plt.legend()
plt.show()

**Insight**<br>
Anime production experienced rapid expansion until 2016–2018, followed by a contraction in 2019–2020, and a recovery phase after 2022.

**T3 — Quality Trend**

In [0]:
score_trend = scored.groupby("year")["score"].mean().reset_index()
score_trend.columns = ["year", "avg_score"]

score_trend["rolling"] = score_trend["avg_score"].rolling(3).mean()

# Peak & trough
peak_year = score_trend.loc[score_trend["avg_score"].idxmax()]
low_year = score_trend.loc[score_trend["avg_score"].idxmin()]

print(f"\nBest year: {peak_year['year']} ({peak_year['avg_score']:.2f})")
print(f"Worst year: {low_year['year']} ({low_year['avg_score']:.2f})")

**Insight**
There is an inverse pattern between production and quality — quality declined during peak production years and improved as production stabilized.

**T4 — Plot Quality Trend**

In [0]:
plt.figure(figsize=(10,5))

plt.plot(score_trend["year"], score_trend["avg_score"], label="Avg Score")
plt.plot(score_trend["year"], score_trend["rolling"], linestyle="--", label="3Y Avg")

plt.axvline(peak_year["year"], linestyle=":")
plt.axvline(low_year["year"], linestyle=":")

plt.title("Anime Quality Over Time")
plt.xlabel("Year")
plt.ylabel("Score")
plt.legend()
plt.show()

**Insights**<br>
There is an inverse pattern between production and quality — quality declined during peak production years and improved as production stabilized.

**T5 — Genre Growth**

In [0]:
df_exp = ts_df.assign(
    genres=ts_df["genres"].str.split(",")
).explode("genres")

df_exp["genres"] = df_exp["genres"].str.strip()

early = df_exp[(df_exp["year"] <= 2017)].groupby("genres")["title"].count()
late = df_exp[(df_exp["year"] >= 2018)].groupby("genres")["title"].count()

growth = pd.DataFrame({
    "early": early,
    "late": late
}).dropna()

growth["growth_%"] = ((growth["late"] - growth["early"]) / growth["early"]) * 100

growth = growth.sort_values("growth_%", ascending=False)

print("\nGenre Growth\n")
print(growth.head(10))

**Insights**<br>
Fantasy and Adventure genres have seen the strongest expansion, indicating a shift toward high-concept storytelling, while Romance has remained relatively stable.